In [3]:
import pandas as pd
import numpy as np

# 재현성 고정
np.random.seed(42)

# 데이터 로드
orders = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\orders.csv")
inventory = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\inventory.csv")
mfg = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\00_used_data\hybrid_manufacturing_categorical.csv")

print("orders:\n", orders.head())
print("inventory:\n", inventory.head())
print("mfg:\n", mfg.head())

orders:
     order_id product_id  order_qty  order_date    due_date
0  ORD000001     PRD012         35  2025-11-19  2025-11-27
1  ORD000002     PRD002         87  2025-10-17  2025-11-04
2  ORD000003     PRD004         50  2025-08-28  2025-09-08
3  ORD000004     PRD008         65  2025-09-23  2025-10-11
4  ORD000005     PRD009         37  2025-11-02  2025-11-16
inventory:
   material_id  unit_price  stock_qty  incoming_qty  lead_time  \
0        SKU0   69.808006         58            96         29   
1        SKU1   14.843523         53            37         23   
2        SKU2   11.319683          1            88         12   
3        SKU3   61.163343         23            59         24   
4        SKU4    4.805496          5            56          5   

   manufacturing_cost  
0           46.279879  
1           33.616769  
2           30.688019  
3           35.624741  
4           92.065161  
mfg:
   Job_ID Machine_ID Operation_Type  Material_Used  Processing_Time  \
0   J001      

In [ ]:
# 공정별 표준 사이클타임
cycle_time_map = (
    mfg.groupby("Operation_Type")["Processing_Time"].mean().round(2).to_dict()
)

print("cycle_time_map:", cycle_time_map)

cycle_time_map: {'Additive': 70.61052631578947, 'Drilling': 71.47089947089947, 'Grinding': 73.36538461538461, 'Lathe': 71.16981132075472, 'Milling': 70.2089552238806}


In [7]:
# product_id 정의
product_ids = sorted(orders["product_id"].unique())

# material_id 정의
material_ids = sorted(inventory["material_id"].unique())

# 공정 정의
process_master = [
    ("A", 1, "Milling"),
    ("B", 2, "Lathe"),
    ("C", 3, "Drilling"),
    ("D", 4, "Grinding"),
    ("E", 5, "Additive")
]

# 공정별 원자재 필요 수량 범위 정의
required_qty_range = {
    "Milling" : (2, 5),
    "Lathe" : (2, 4),
    "Drilling" : (1, 3),
    "Grinding" : (1, 2),
    "Additive" : (3, 6)
}

In [ ]:
# 각 행을 생성
rows = []

for product_id in product_ids:
    # 동일 제품 안에서 원자재 과대 중복 방지
    used_materials = set()

    for process_id, process_step, process_name in process_master:
        # 공정별 원자재 3~5개 사용
        n_materials = np.random.randint(3, 6)

        # 아직 사용하지 않은 원자재를 우선 후보로 사용
        available_materials = [m for m in material_ids if m not in used_materials]

        # 후보 원자재가 부족하다면 전체 원자재에서 다시 선택
        if len(available_materials) < n_materials:
            available_materials = material_ids
        
        # 해당 제품/공정에서 사용할 원자재 선택
        selected_materials = np.random.choice(
            available_materials,
            size=n_materials,
            replace=False
        )

        # 공정별 표준 사이클 타임
        standard_cycle_time = cycle_time_map.get(process_name)

        # 공정별 필요 수량 범위
        qty_low, qty_high = required_qty_range[process_name]

        for material_id in selected_materials: 
            # 현재 제품에서 사용한 원자재 기록
            used_materials.add(material_id)

            # 공정 특성에 맞는 원자재 필요 수량 생성
            required_material_qty = np.random.randint(qty_low, qty_high+1)

            rows.append({
                "product_id" : product_id,
                "process_id" : process_id,
                "material_id" : material_id,
                "process_step" : process_step,
                "process_name" : process_name,
                "required_material_qty" : required_material_qty,
                "standard_cycle_time" : standard_cycle_time
            })

# DataFrame으로 변환
raw_process = pd.DataFrame(rows)

raw_process.head()

,product_id,process_id,material_id,process_step,process_name,required_material_qty,standard_cycle_time
0,PRD001,A,SKU84,1,Milling,4,70.208955
1,PRD001,A,SKU57,1,Milling,2,70.208955
2,PRD001,A,SKU72,1,Milling,5,70.208955
3,PRD001,A,SKU5,1,Milling,4,70.208955
4,PRD001,A,SKU49,1,Milling,4,70.208955


In [9]:
# QC
# ============================================================
# raw_process QC
# ============================================================

print("===== 기본 정보 =====")
print("row 수:", len(raw_process))
print("columns:", raw_process.columns.tolist())

print("\n===== PK 체크 =====")
print(
    "PK 중복:",
    raw_process.duplicated(["product_id", "process_id", "material_id"]).sum()
)

print("\n===== 제품/공정 체크 =====")
print("제품 수:", raw_process["product_id"].nunique())
print("공정 수:", raw_process["process_id"].nunique())

print("\n제품별 공정 수:")
print(raw_process.groupby("product_id")["process_id"].nunique().describe())

print("\n제품-공정별 원자재 수:")
print(
    raw_process
    .groupby(["product_id", "process_id"])["material_id"]
    .nunique()
    .describe()
)

print("\n===== FK 체크 =====")
print("material_id FK 정상 여부:", raw_process["material_id"].isin(inventory["material_id"]).all())
print("product_id FK 정상 여부:", raw_process["product_id"].isin(orders["product_id"]).all())

print("\n===== 사이클타임 체크 =====")
print("standard_cycle_time 결측:", raw_process["standard_cycle_time"].isna().sum())
print(raw_process.groupby(["process_id", "process_name"])["standard_cycle_time"].first())

print("\n===== 필요 원자재 수량 체크 =====")
print(raw_process["required_material_qty"].describe())

print("\n공정별 필요 원자재 수량 범위:")
print(
    raw_process
    .groupby("process_name")["required_material_qty"]
    .agg(["min", "max", "mean"])
)

===== 기본 정보 =====
row 수: 394
columns: ['product_id', 'process_id', 'material_id', 'process_step', 'process_name', 'required_material_qty', 'standard_cycle_time']

===== PK 체크 =====
PK 중복: 0

===== 제품/공정 체크 =====
제품 수: 20
공정 수: 5

제품별 공정 수:
count    20.0
mean      5.0
std       0.0
min       5.0
25%       5.0
50%       5.0
75%       5.0
max       5.0
Name: process_id, dtype: float64

제품-공정별 원자재 수:
count    100.000000
mean       3.940000
std        0.874094
min        3.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: material_id, dtype: float64

===== FK 체크 =====
material_id FK 정상 여부: True
product_id FK 정상 여부: True

===== 사이클타임 체크 =====
standard_cycle_time 결측: 0
process_id  process_name
A           Milling         70.208955
B           Lathe           71.169811
C           Drilling        71.470899
D           Grinding        73.365385
E           Additive        70.610526
Name: standard_cycle_time, dtype: float64

===== 필요 원자재 수량 체크 =====
cou

In [10]:
# CSV로 내보내기
output_path = r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\process.csv"

try:
    raw_process.to_csv(output_path, index=False, encoding='utf-8-sig')
    print("process.csv 생성 완료:", raw_process.shape)
    
except Exception as e:
    print("생성 실패:", e)

process.csv 생성 완료: (394, 7)
